# Homework 1 — Question 3

Implement the robust sparse regression MILP from Question 2 (Part B) and run it on `sublets.csv`.

**Instructions**
1. Place `sublets.csv` in the same directory as this notebook.
2. Fill in `solve_robust_sparse` below.
3. Run the cells for parts (a)–(c) and report the printed outputs on your written solutions.


## Imports


In [ ]:
using JuMP, Gurobi, CSV, DataFrames, LinearAlgebra, Statistics


## Load and preprocess data

We center `rent` and standardize every feature column (no intercept).


In [ ]:
df = CSV.read("sublets.csv", DataFrame)
feature_names = names(df)[2:end]   # all columns except rent

y = Vector{Float64}(df.rent)
y = y .- mean(y)

X = Matrix{Float64}(df[:, Not(:rent)])
X = (X .- mean(X, dims=1)) ./ std(X, dims=1)

n, p = size(X)
println("n = $n, p = $p")
println("features: ", feature_names)


## The MILP solver

Complete the function below. It should implement your model from Question 2 Part B.

Return a `NamedTuple` with fields `obj`, `u`, `w`, `β`, `s`.

In [ ]:
function solve_robust_sparse(X, y, λ, ρ, k; M=1e3, solver_output=0)
    n, p = size(X)
    @assert length(y) == n

    model = Model(Gurobi.Optimizer)
    set_optimizer_attribute(model, "OutputFlag", solver_output)

    ################# ADD YOUR CODE AFTER THIS LINE #######################

    # Hint: create variables β[1:p], s[1:p] (Bin), t[1:p], u, w,
    # set the objective, add the constraints from Question 2 / the markdown
    # above, then call optimize!(model).

    # Placeholder so the notebook parses before you fill things in:
    β_val = zeros(p)
    s_val = zeros(p)
    u_val = 0.0
    w_val = 0.0
    obj_val = 0.0

    ################# ADD YOUR CODE BEFORE THIS LINE #######################

    return (obj=obj_val, u=u_val, w=w_val, β=β_val, s=s_val)
end


In [ ]:
"""Pretty-print a solution."""
function report_solution(sol; feature_names=feature_names, atol=1e-6)
    println("objective = $(round(sol.obj, digits=4))")
    println("u         = $(round(sol.u, digits=4))")
    println("w         = $(round(sol.w, digits=4))")
    selected = findall(j -> sol.s[j] > 0.5 && abs(sol.β[j]) > atol, 1:length(sol.β))
    println("selected features:")
    for j in selected
        println("  $(feature_names[j]): β = $(round(sol.β[j], digits=4))")
    end
    if isempty(selected)
        println("  (none)")
    end
    return selected
end


## Part (a)

Solve with $\lambda = 0.25$, $\rho = 0.5$, $k = 3$. Report the objective, $u$, and selected coefficients.


In [ ]:
sol_a = solve_robust_sparse(X, y, 0.25, 0.5, 3)
report_solution(sol_a);


## Part (b)

Compare three uncertainty regimes with $k = 3$ fixed.


In [ ]:
cases_b = [
    ("A only (λ=0.25, ρ=0)", 0.25, 0.0),
    ("B only (λ=0, ρ=1)",    0.0,  1.0),
    ("both   (λ=0.25, ρ=0.5)", 0.25, 0.5),
]

for (label, λ, ρ) in cases_b
    println("="^60)
    println(label)
    sol = solve_robust_sparse(X, y, λ, ρ, 3)
    report_solution(sol)
    println()
end


## Part (c)

Compare sparse ($k = 3$) vs dense ($k = 7$) with $\lambda = 0.25$, $\rho = 0.5$.


In [ ]:
println("===== k = 3 (sparse) =====")
sol_sparse = solve_robust_sparse(X, y, 0.25, 0.5, 3)
report_solution(sol_sparse)
println()

println("===== k = 7 (dense) =====")
sol_dense = solve_robust_sparse(X, y, 0.25, 0.5, 7)
report_solution(sol_dense);
